# FreshFlow Juice Company - NumPy Data Analyst Project
**Module 07 | NumPy only (no Pandas)**

Phases: 1 Load  2 Filter  3 Clean  4 Metrics  5 Statistics  6 Compare  7 Daily Trend  8 Anomalies  9 Discounts  10 Pipeline  11 Tests  12 Report

**Data Dictionary (D1)**

| # | Column | Meaning | Allowed |
|---|---|---|---|
| 0 | transaction_id | Bill number | Unique |
| 1 | day_index | Day | 1-30 |
| 2 | outlet_id | Outlet | 1 A, 2 B, 3 C |
| 3 | product_code | Product | 1-8 |
| 4 | category_code | Category | 1 Juice, 2 Smoothie, 3 Lemonade, 4 Tea |
| 5 | quantity | Units sold | > 0 |
| 6 | unit_price | Price (INR) | > 0 |
| 7 | unit_cost | Cost (INR) | >= 0 |
| 8 | discount_rate | 0.15 = 15% | 0-1 |
| 9 | stock_before_units | Stock | >= 0 |
| 10 | payment_code | Payment | 1 Cash, 2 UPI, 3 Card, 4 Other |
| 11 | customer_rating | Rating | 1-5 or missing |

Products 1-4 = Juice (Orange, Mango, Apple, Watermelon), 5-6 = Smoothie (Banana, Mixed Fruit), 7 = Lemonade, 8 = Iced Tea

---
## Phase 1 - Load and Inspect
**1. Import Libraries and Load File**

In [64]:
import numpy as np
np.set_printoptions(suppress=True, linewidth=120)
data = np.genfromtxt("freshflow_sales.csv", delimiter=",", skip_header=1, ndmin=2)
raw_data = data.copy()      # original data kept safe
print(data[:5])

[[100001.        1.        1.        6.        2.        2.      128.       59.94      0.05     39.        3.
       5.  ]
 [100002.        1.        2.        6.        2.        3.      129.       70.23      0.       44.        3.
       5.  ]
 [100003.        1.        3.        2.        1.        5.       98.       40.81      0.       61.        1.
       5.  ]
 [100004.        1.        1.        1.        1.        6.       77.       37.36      0.1     117.        4.
       3.  ]
 [100005.        1.        1.        2.        1.        3.       92.       41.51      0.      129.        2.
       5.  ]]


**Helper Functions** (used again and again in the project)

In [65]:
def group_total(codes, values=None, size=0):
    """Add values for each code (1, 2, 3 ...) and drop the empty 0 slot."""
    return np.bincount(codes.astype(int), weights=values, minlength=size)[1:]

def margin_percent(profit, revenue):
    """Profit as % of revenue."""
    return profit / revenue * 100

def z_score(values):
    """How many standard deviations each value is from the mean."""
    return (values - values.mean()) / values.std()

def show_histogram(values, bins=5):
    """Print histogram frequency and bin edges."""
    freq, edges = np.histogram(values, bins=bins)
    print("Frequency:", freq)
    print("Bins:", np.round(edges))

def save_file(name, lines):
    """Write a list of lines to a text file."""
    with open(name, "w") as file:
        file.write("\n".join(lines) + "\n")

**2. Test Loader on Small Example** (blank should become NaN)

In [66]:
from io import StringIO
test = np.genfromtxt(StringIO("a,b\n1,2\n3,\n"), delimiter=",", skip_header=1, ndmin=2)
print("Small test:\n", test, "\nBlank became NaN:", np.isnan(test[1, 1]))
print("loadtxt on clean data:\n", np.loadtxt(StringIO("x,y\n1,2\n3,4\n"), delimiter=",", skiprows=1))

Small test:
 [[ 1.  2.]
 [ 3. nan]] 
Blank became NaN: True
loadtxt on clean data:
 [[1. 2.]
 [3. 4.]]


**3. Dataset Information**

In [67]:
print("Shape:", data.shape, "Size:", data.size, "Dimensions:", data.ndim)
print("Data Type:", data.dtype, "Item Size:", data.itemsize, "Bytes:", data.nbytes)
assert data.shape[0] == 5000
print("5000 rows loaded correctly")

Shape: (5000, 12) Size: 60000 Dimensions: 2
Data Type: float64 Item Size: 8 Bytes: 480000
5000 rows loaded correctly


**4. First and Last Records**

In [68]:
print("First 5 Rows:\n", data[:5])
print("Last 5 Rows:\n", data[-5:])
print("Infinite Values:", np.isinf(data).sum())

First 5 Rows:
 [[100001.        1.        1.        6.        2.        2.      128.       59.94      0.05     39.        3.
       5.  ]
 [100002.        1.        2.        6.        2.        3.      129.       70.23      0.       44.        3.
       5.  ]
 [100003.        1.        3.        2.        1.        5.       98.       40.81      0.       61.        1.
       5.  ]
 [100004.        1.        1.        1.        1.        6.       77.       37.36      0.1     117.        4.
       3.  ]
 [100005.        1.        1.        2.        1.        3.       92.       41.51      0.      129.        2.
       5.  ]]
Last 5 Rows:
 [[104996.       30.        2.        7.        3.        3.       52.       26.76      0.       92.        2.
       3.  ]
 [104997.       30.        1.        5.        2.        1.      118.       54.81      0.       91.        2.
       5.  ]
 [104998.       30.        1.        1.        1.        1.       77.       38.61      0.1      55.        3.

**5. Missing Values and Range**

In [69]:
columns = ["ID", "Day", "Outlet", "Product", "Category", "Quantity", "Price", "Cost", "Discount", "Stock", "Payment", "Rating"]
print("Missing:", dict(zip(columns, np.isnan(data).sum(axis=0).tolist())))
print("Min:", np.nanmin(data, axis=0))
print("Max:", np.nanmax(data, axis=0))

Missing: {'ID': 0, 'Day': 0, 'Outlet': 0, 'Product': 0, 'Category': 0, 'Quantity': 0, 'Price': 4, 'Cost': 4, 'Discount': 0, 'Stock': 0, 'Payment': 0, 'Rating': 20}
Min: [100001.        1.        1.        1.        1.       -2.      -94.      -43.04      0.        0.        1.
      0.  ]
Max: [105000.       30.        9.        8.        4.       10.     1048.       73.01      1.5     149.        4.
      7.  ]


**6. Python List vs NumPy Array**

In [70]:
price_list = [80, 90, 100]
price_array = np.array(price_list)
print("List * 2:", price_list * 2)       # list repeats
print("Array * 2:", price_array * 2)     # values double

List * 2: [80, 90, 100, 80, 90, 100]
Array * 2: [160 180 200]


**7. Array Creation**

In [71]:
arr = np.array([10, 20, 30, 40])
print(np.asarray(arr), np.arange(1, 10), np.linspace(1, 10, 5))
print(np.zeros(3), np.ones(3), np.full(3, 10))
print(np.eye(3))

[10 20 30 40] [1 2 3 4 5 6 7 8 9] [ 1.    3.25  5.5   7.75 10.  ]
[0. 0. 0.] [1. 1. 1.] [10 10 10]
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


**8. Array Properties and astype()**

In [72]:
print("Shape:", arr.shape, "ndim:", arr.ndim, "Size:", arr.size)
print("dtype:", arr.dtype, "itemsize:", arr.itemsize, "nbytes:", arr.nbytes)
float_arr = arr.astype(float)
print("astype(float):", float_arr, "itemsize:", float_arr.itemsize)

Shape: (4,) ndim: 1 Size: 4
dtype: int64 itemsize: 8 nbytes: 32
astype(float): [10. 20. 30. 40.] itemsize: 8


---
## Phase 2 - Select and Filter
**9. Indexing and Slicing**

In [73]:
print("First Row:", data[0])
print("Last Row:", data[-1])
print("First Column (5 values):", data[:5, 0])
print("Rows 0-2, Columns 5-8:\n", data[0:3, 5:9])

First Row: [100001.        1.        1.        6.        2.        2.      128.       59.94      0.05     39.        3.
      5.  ]
Last Row: [105000.       30.        2.        7.        3.        4.       48.       21.63      0.       22.        2.
      5.  ]
First Column (5 values): [100001. 100002. 100003. 100004. 100005.]
Rows 0-2, Columns 5-8:
 [[  2.   128.    59.94   0.05]
 [  3.   129.    70.23   0.  ]
 [  5.    98.    40.81   0.  ]]


**10. Boolean Masks**

In [74]:
print("Outlet 1 Rows:", (data[:, 2] == 1).sum())
print("High Discount (>25%):\n", data[data[:, 8] > 0.25][:, [0, 5, 6, 8]])
print("Invalid Quantity:\n", data[data[:, 5] <= 0][:, [0, 5, 6]])

Outlet 1 Rows: 2010
High Discount (>25%):
 [[100259.        3.       82.        1.2 ]
 [101237.        3.       83.        0.9 ]
 [102656.        3.       71.        0.6 ]
 [103008.        5.       87.        1.5 ]
 [103397.        4.       82.        0.7 ]
 [103840.        2.       67.        0.65]
 [104616.        5.       83.        0.8 ]]
Invalid Quantity:
 [[101401.     -2.     49.]
 [104246.     -1.    121.]
 [104556.      0.    119.]
 [104698.      0.     48.]]


**11. Compound Conditions (& | ~)**

In [75]:
print("Outlet 1 AND Product 1:", ((data[:, 2] == 1) & (data[:, 3] == 1)).sum())
print("Product 5 OR 6:", ((data[:, 3] == 5) | (data[:, 3] == 6)).sum())
print("NOT Cash:", (~(data[:, 10] == 1)).sum())

Outlet 1 AND Product 1: 401
Product 5 OR 6: 1049
NOT Cash: 4022


**12. np.where() and np.nonzero()**

In [76]:
print("High Discount Rows:", np.where(data[:, 8] > 0.25)[0])
print("Same with nonzero:", np.nonzero(data[:, 8] > 0.25)[0])
print(np.where(data[:, 8] > 0.20, "High", "Low")[:10])

High Discount Rows: [ 258 1236 2655 3007 3396 3839 4615]
Same with nonzero: [ 258 1236 2655 3007 3396 3839 4615]
['Low' 'Low' 'Low' 'Low' 'Low' 'Low' 'Low' 'Low' 'Low' 'Low']


**13. np.select()**

In [77]:
conditions = [data[:, 5] <= 0, data[:, 5] <= 2, data[:, 5] <= 5]
labels = np.select(conditions, ["Invalid", "Low", "Medium"], default="High")
print(np.unique(labels, return_counts=True))

(array(['High', 'Invalid', 'Low', 'Medium'], dtype='<U7'), array([ 182,    4, 2448, 2366]))


---
## Phase 3 - Clean and Validate
**14. Separate Columns**

In [78]:
ids, day, outlet, product, category = data[:, 0], data[:, 1], data[:, 2], data[:, 3], data[:, 4]
quantity, price, cost, discount = data[:, 5], data[:, 6], data[:, 7], data[:, 8]
stock, payment, rating = data[:, 9], data[:, 10], data[:, 11]
print("Code columns are whole numbers:", np.all(data[:, :6] == np.round(data[:, :6])))

Code columns are whole numbers: True


**15. Missing and Wrong Values**

In [79]:
missing_price, missing_cost = np.isnan(price), np.isnan(cost)
bad_quantity, bad_price, bad_cost = quantity <= 0, price <= 0, cost < 0
bad_discount = (discount < 0) | (discount > 1)
print("Missing Price:", missing_price.sum(), "Missing Cost:", missing_cost.sum())
print("Bad Qty:", bad_quantity.sum(), "Bad Price:", bad_price.sum(), "Bad Cost:", bad_cost.sum(), "Bad Discount:", bad_discount.sum())

Missing Price: 4 Missing Cost: 4
Bad Qty: 4 Bad Price: 2 Bad Cost: 2 Bad Discount: 2


**16. Wrong Codes**

In [80]:
bad_outlet = ~np.isin(outlet, [1, 2, 3])
bad_day = ~np.isin(day, range(1, 31))
bad_payment = ~np.isin(payment, [1, 2, 3, 4])
print("Unknown Outlet:", bad_outlet.sum(), "Bad Day:", bad_day.sum(), "Bad Payment:", bad_payment.sum())

Unknown Outlet: 2 Bad Day: 0 Bad Payment: 0


**17. Product / Category Check**

In [81]:
category_of = np.array([0, 1, 1, 1, 1, 2, 2, 3, 4])      # position = product code
good_product = np.isin(product, range(1, 9))
bad_category = good_product & (category != category_of[np.where(good_product, product, 0).astype(int)])
print("Unknown Product:", (~good_product).sum(), "Category Mismatch:", bad_category.sum())

Unknown Product: 0 Category Mismatch: 4


**18. Rating Check** (missing is allowed, never made 0)

In [82]:
missing_rating = np.isnan(rating)
bad_rating = (rating < 1) | (rating > 5)
print("Missing Rating:", missing_rating.sum(), "Rating outside 1-5:", bad_rating.sum())

Missing Rating: 20 Rating outside 1-5: 3


**19. Duplicate Transaction IDs**

In [83]:
unique_ids, id_counts = np.unique(ids, return_counts=True)
duplicate = np.isin(ids, unique_ids[id_counts > 1])
print("Duplicate IDs:", unique_ids[id_counts > 1], "Rows:", duplicate.sum())
print(data[duplicate][:, [0, 1, 2, 3, 5, 6]])

Duplicate IDs: [100794. 104403.] Rows: 5
[[100794.      3.      2.      4.      3.     70.]
 [100794.      5.      1.      8.      5.     62.]
 [100794.     12.      2.      3.      1.     84.]
 [104403.     24.      3.      4.      3.     72.]
 [104403.     27.      2.      7.      3.     48.]]


The duplicate rows have different day, outlet and product, so they are different sales. We **keep** them and only flag them.

**20. Suspicious but Possible** (keep, only flag)

In [84]:
high_price = price > 3 * np.nanmedian(price)
high_discount = (discount > 0.5) & (discount <= 1)
zero_stock = (stock == 0) & (quantity > 0)
print("High Price:", high_price.sum(), "High Discount:", high_discount.sum(), "Zero Stock:", zero_stock.sum())

High Price: 3 High Discount: 5 Zero Stock: 5


**21. Create Clean Data**

In [85]:
invalid = missing_price | missing_cost | bad_quantity | bad_price | bad_cost
invalid = invalid | bad_discount | bad_outlet | bad_category | bad_day | bad_payment
clean_data = data[~invalid].copy()
clean_rating = np.where((clean_data[:, 11] >= 1) & (clean_data[:, 11] <= 5), clean_data[:, 11], np.nan)
print("Total:", len(data), "Removed:", invalid.sum(), "Valid:", len(clean_data))

Total: 5000 Removed: 24 Valid: 4976


**Cleaning rules:** invalid rows are removed from calculations; suspicious rows are kept and flagged;
missing prices/costs are not filled; wrong ratings become NaN; `raw_data` is never changed.

---
## Phase 4 - Sales and Profit
**22. Metric Function**

In [86]:
def calculate(qty, price, cost, disc):
    gross = qty * price               # gross sales
    disc_amount = gross * disc        # discount in rupees
    net = gross - disc_amount         # net revenue
    total_cost = qty * cost
    profit = net - total_cost         # gross profit
    return gross, disc_amount, net, total_cost, profit

**23. Calculate Sales and Profit**

In [87]:
quantity, price, cost, discount = clean_data[:, 5], clean_data[:, 6], clean_data[:, 7], clean_data[:, 8]
gross_sales, discount_amount, net_revenue, total_cost, gross_profit = calculate(quantity, price, cost, discount)
print("Gross Sales:", round(gross_sales.sum(), 2), "Discount:", round(discount_amount.sum(), 2))
print("Net Revenue:", round(net_revenue.sum(), 2), "Cost:", round(total_cost.sum(), 2))
print("Gross Profit:", round(gross_profit.sum(), 2))

Gross Sales: 1169589.0 Discount: 79333.45
Net Revenue: 1090255.55 Cost: 555035.62
Gross Profit: 535219.93


**24. Gross Margin**

In [88]:
margin = margin_percent(gross_profit.sum(), net_revenue.sum())
bill_margin = np.where(net_revenue != 0, margin_percent(gross_profit, net_revenue), np.nan)
print("Gross Margin:", round(margin, 2), "%", "| Loss Bills:", (gross_profit < 0).sum())

Gross Margin: 49.09 % | Loss Bills: 5


**25. Other Metrics**

In [89]:
print("Avg Selling Price:", round((net_revenue / quantity).mean(), 2))
print("Avg Transaction Value:", round(net_revenue.sum() / len(clean_data), 2))
print("Weighted Discount %:", round(discount_amount.sum() / gross_sales.sum() * 100, 2))
print("Units Sold:", quantity.sum())

Avg Selling Price: 81.01
Avg Transaction Value: 219.1
Weighted Discount %: 6.78
Units Sold: 13593.0


---
## Phase 5 - Statistics
**26. Statistical Analysis**

In [90]:
print("Mean:", round(np.mean(net_revenue), 2), "Median:", np.median(net_revenue))
print("Min:", round(np.min(net_revenue), 2), "Max:", np.max(net_revenue), "Range:", round(np.ptp(net_revenue), 2))
print("Std:", round(np.std(net_revenue), 2), "Sample Std:", round(np.std(net_revenue, ddof=1), 2), "Var:", round(np.var(net_revenue), 2))
print("Quartiles:", np.percentile(net_revenue, [25, 50, 75]), "90% Quantile:", np.quantile(net_revenue, 0.9))

Mean: 219.1 Median: 194.0
Min: 24.9 Max: 1991.2 Range: 1966.3
Std: 129.52 Sample Std: 129.53 Var: 16775.13
Quartiles: [126.775 194.    276.   ] 90% Quantile: 384.0


Mean > median, so a few big bills pull the average up (right skew).

**27. Weighted Average and Histogram**

In [91]:
print("Simple Avg Price:", round(np.mean(price), 2), "Weighted Avg Price:", round(np.average(price, weights=quantity), 2))
show_histogram(net_revenue)

Simple Avg Price: 86.15 Weighted Avg Price: 86.04
Frequency: [4612  358    4    1    1]
Bins: [  25.  418.  811. 1205. 1598. 1991.]


**28. Customer Rating**

In [92]:
print("Valid Ratings:", (~np.isnan(clean_rating)).sum(), "Missing/Invalid:", np.isnan(clean_rating).sum())
print("Average Rating:", round(np.nanmean(clean_rating), 2))
print("Count of 1 to 5:", [int((clean_rating == i).sum()) for i in range(1, 6)])

Valid Ratings: 4953 Missing/Invalid: 23
Average Rating: 3.82
Count of 1 to 5: [206, 369, 1004, 1910, 1464]


---
## Phase 6 - Compare Outlets and Products
**29. Outlet-Wise Sales**

In [93]:
outlet_names = ["Outlet A", "Outlet B", "Outlet C"]
outlet_code = clean_data[:, 2]
outlet_sales = group_total(outlet_code, net_revenue)
outlet_profit = group_total(outlet_code, gross_profit)
print("Bills:", group_total(outlet_code), "\nRevenue:", outlet_sales, "\nProfit:", outlet_profit)
print("Margin %:", np.round(margin_percent(outlet_profit, outlet_sales), 2))

Bills: [2003 1721 1252] 
Revenue: [438666.4  376609.15 274980.  ] 
Profit: [216289.73 183843.71 135086.49]
Margin %: [49.31 48.82 49.13]


**30. Product-Wise Totals**

In [94]:
product_names = ["Orange Juice", "Mango Juice", "Apple Juice", "Watermelon Juice", "Banana Smoothie", "Mixed Fruit Smoothie", "Lemonade", "Iced Tea"]
product_code = clean_data[:, 3]
product_units = group_total(product_code, quantity)
product_sales = group_total(product_code, net_revenue)
product_profit = group_total(product_code, gross_profit)

**31. Product-Wise Sales**

In [95]:
for i in range(8):
    print(product_names[i], "Units:", product_units[i], "Revenue:", round(product_sales[i], 2),
          "Profit:", round(product_profit[i], 2), "Per Unit:", round(product_profit[i] / product_units[i], 2))

Orange Juice Units: 2663.0 Revenue: 198791.85 Profit: 97181.5 Per Unit: 36.49
Mango Juice Units: 2315.0 Revenue: 205306.6 Profit: 100755.57 Per Unit: 43.52
Apple Juice Units: 1378.0 Revenue: 109371.6 Profit: 53252.43 Per Unit: 38.64
Watermelon Juice Units: 1545.0 Revenue: 101971.15 Profit: 50518.2 Per Unit: 32.7
Banana Smoothie Units: 1385.0 Revenue: 155863.0 Profit: 76577.16 Per Unit: 55.29
Mixed Fruit Smoothie Units: 1411.0 Revenue: 171632.5 Profit: 84260.79 Per Unit: 59.72
Lemonade Units: 1633.0 Revenue: 76542.7 Profit: 37734.9 Per Unit: 23.11
Iced Tea Units: 1263.0 Revenue: 70776.15 Profit: 34939.38 Per Unit: 27.66


**32. Category-Wise Sales**

In [96]:
category_names = ["Juice", "Smoothie", "Lemonade", "Tea"]
category_code = clean_data[:, 4]
print("Revenue:", dict(zip(category_names, np.round(group_total(category_code, net_revenue)).tolist())))
print("Profit:", dict(zip(category_names, np.round(group_total(category_code, gross_profit)).tolist())))

Revenue: {'Juice': 615441.0, 'Smoothie': 327495.0, 'Lemonade': 76543.0, 'Tea': 70776.0}
Profit: {'Juice': 301708.0, 'Smoothie': 160838.0, 'Lemonade': 37735.0, 'Tea': 34939.0}


**33. Sorting and Ranking**

In [97]:
print("Outlet Ranking:", [outlet_names[i] for i in np.argsort(outlet_sales)[::-1]])
print("Units Ranking:", [product_names[i] for i in np.argsort(product_units)[::-1]])
print("Profit Ranking:", [product_names[i] for i in np.argsort(product_profit)[::-1]])
print("Best Outlet:", outlet_names[np.argmax(outlet_sales)], "| Least Profit:", product_names[np.argmin(product_profit)])

Outlet Ranking: ['Outlet A', 'Outlet B', 'Outlet C']
Units Ranking: ['Orange Juice', 'Mango Juice', 'Lemonade', 'Watermelon Juice', 'Mixed Fruit Smoothie', 'Banana Smoothie', 'Apple Juice', 'Iced Tea']
Profit Ranking: ['Mango Juice', 'Orange Juice', 'Mixed Fruit Smoothie', 'Banana Smoothie', 'Apple Juice', 'Watermelon Juice', 'Lemonade', 'Iced Tea']
Best Outlet: Outlet A | Least Profit: Iced Tea


Orange Juice sells the most units, but Mango Juice makes the most profit, because profit per unit matters, not just units.

**34. Outlet x Product Table**

In [98]:
sales_table = np.zeros((3, 8))
np.add.at(sales_table, (outlet_code.astype(int) - 1, product_code.astype(int) - 1), net_revenue)
print("Revenue Table (outlets x products):\n", np.round(sales_table))
print("Outlet Totals:", sales_table.sum(axis=1), "\nProduct Totals:", sales_table.sum(axis=0), "\nTranspose Shape:", sales_table.T.shape)

Revenue Table (outlets x products):
 [[81223. 84095. 43452. 43451. 58450. 72283. 28635. 27076.]
 [68433. 71168. 37875. 31571. 59342. 54997. 28481. 24742.]
 [49135. 50043. 28044. 26949. 38070. 44353. 19427. 18958.]]
Outlet Totals: [438666.4  376609.15 274980.  ] 
Product Totals: [198791.85 205306.6  109371.6  101971.15 155863.   171632.5   76542.7   70776.15] 
Transpose Shape: (8, 3)


**35. Payment Method Analysis**

In [99]:
payment_names = ["Cash", "UPI", "Card", "Other"]
payment_code = clean_data[:, 10]
print("Bills:", dict(zip(payment_names, group_total(payment_code).tolist())))
print("Revenue:", dict(zip(payment_names, np.round(group_total(payment_code, net_revenue)).tolist())))

Bills: {'Cash': 974, 'UPI': 2534, 'Card': 986, 'Other': 482}
Revenue: {'Cash': 211734.0, 'UPI': 552163.0, 'Card': 218093.0, 'Other': 108266.0}


---
## Phase 7 - Daily Trend
**36. Day-Wise Sales**

In [100]:
day_code = clean_data[:, 1]
daily_sales = group_total(day_code, net_revenue, size=31)
daily_bills = group_total(day_code, size=31).astype(int)
print("Daily Sales:", np.round(daily_sales))
print("Daily Bills:", daily_bills)

Daily Sales: [33841. 36719. 34714. 37014. 42641. 34926. 36106. 35662. 38926. 39691. 29969. 65330. 38825. 36717. 39694. 35785. 32762.
 35863. 34448. 35548. 31127. 16096. 35861. 37165. 32616. 38953. 36562. 37703. 32836. 36155.]
Daily Bills: [152 164 167 172 183 169 165 166 177 174 147 310 175 172 181 162 149 168 155 156 146  73 170 168 151 174 148 168 152
 162]


**37. Daily Sales Change**

In [101]:
daily_change = np.diff(daily_sales)
percent_change = np.where(daily_sales[:-1] != 0, daily_change / daily_sales[:-1] * 100, np.nan)
print("Daily Change:", np.round(daily_change))
print("Percent Change:", np.round(percent_change, 1))

Daily Change: [  2878.  -2004.   2300.   5627.  -7715.   1181.   -445.   3265.    765.  -9722.  35361. -26505.  -2108.   2977.
  -3909.  -3023.   3101.  -1415.   1100.  -4421. -15030.  19765.   1304.  -4549.   6337.  -2390.   1141.  -4867.
   3318.]
Percent Change: [  8.5  -5.5   6.6  15.2 -18.1   3.4  -1.2   9.2   2.  -24.5 118.  -40.6  -5.4   8.1  -9.8  -8.4   9.5  -3.9   3.2
 -12.4 -48.3 122.8   3.6 -12.2  19.4  -6.1   3.1 -12.9  10.1]


**38. Moving Average and Cumulative Sales**

In [102]:
moving_average = np.convolve(daily_sales, np.ones(3) / 3, mode="valid")
print("3-Day Moving Average:", np.round(moving_average))
print("Cumulative Sales:", np.round(np.cumsum(daily_sales)))

3-Day Moving Average: [35091. 36149. 38123. 38193. 37891. 35565. 36898. 38093. 36195. 44997. 44708. 46957. 38412. 37399. 36080. 34803. 34358.
 35286. 33708. 27590. 27695. 29707. 35214. 36245. 36044. 37740. 35701. 35565.]
Cumulative Sales: [  33841.   70559.  105274.  142288.  184929.  219854.  255961.  291622.  330549.  370240.  400208.  465538.  504363.
  541081.  580775.  616560.  649322.  685185.  719633.  755181.  786307.  802403.  838264.  875430.  908046.  946998.
  983561. 1021264. 1054101. 1090256.]


**39. Unusual Sales Days**

In [103]:
day_z = z_score(daily_sales)
unusual_days = np.where(np.abs(day_z) > 2)[0] + 1
print("Unusual Days:", unusual_days, "Sales:", np.round(daily_sales[unusual_days - 1]))
print("Bills:", daily_bills[unusual_days - 1], "Avg Bill:", np.round(daily_sales[unusual_days - 1] / daily_bills[unusual_days - 1]))

Unusual Days: [12 22] Sales: [65330. 16096.]
Bills: [310  73] Avg Bill: [211. 220.]


Day 12 has double the bills and day 22 has half, but the average bill is normal, so the change is in the number of customers. With 30 days we cannot talk about seasons.

**40. Daily Sales Histogram**

In [104]:
show_histogram(daily_sales)

Frequency: [ 1 12 16  0  1]
Bins: [16096. 25943. 35790. 45636. 55483. 65330.]


---
## Phase 8 - Anomalies
**41. Outlier Detection (IQR)**

In [105]:
q1, q3 = np.percentile(net_revenue, [25, 75])
iqr = q3 - q1
outliers = (net_revenue < q1 - 1.5 * iqr) | (net_revenue > q3 + 1.5 * iqr)
print("IQR:", iqr, "Outliers:", outliers.sum(), "First IDs:", clean_data[outliers, 0][:5])

IQR: 149.225 Outliers: 176 First IDs: [100059. 100080. 100082. 100113. 100120.]


**42. Z-Score on Price**

In [106]:
price_z = z_score(price)
print("High Price IDs (z > 3):", clean_data[price_z > 3, 0])
print("Their Prices:", price[price_z > 3])

High Price IDs (z > 3): [100310. 100664. 104869.]
Their Prices: [ 648. 1048.  584.]


**43. Very High Discount Bills**

In [107]:
big_discount = discount > 0.5
print("Discount > 50% Bills:", big_discount.sum(), "IDs:", clean_data[big_discount, 0])
print("Their Profit:", gross_profit[big_discount])
print("Their Margin:", round(margin_percent(gross_profit[big_discount].sum(), net_revenue[big_discount].sum()), 2), "%")

Discount > 50% Bills: 5 IDs: [101237. 102656. 103397. 103840. 104616.]
Their Profit: [-82.95 -26.49 -56.84 -12.4  -97.35]
Their Margin: -81.57 %


**44. Exception Register - Flag Bills**

In [108]:
problems = {"MISSING_PRICE": missing_price, "MISSING_COST": missing_cost, "BAD_QTY": bad_quantity, "BAD_PRICE": bad_price,
            "BAD_COST": bad_cost, "BAD_DISCOUNT": bad_discount, "BAD_OUTLET": bad_outlet, "BAD_CATEGORY": bad_category, "BAD_RATING": bad_rating,
            "DUPLICATE": duplicate, "HIGH_PRICE": high_price, "HIGH_DISCOUNT": high_discount, "ZERO_STOCK": zero_stock}
flagged = np.column_stack(list(problems.values())).any(axis=1)
print("Flagged Bills:", flagged.sum())

Flagged Bills: 45


**45. Exception Register - Sort and Save** (biggest bill value first)

In [109]:
bill_value = np.nan_to_num(data[:, 5] * data[:, 6], nan=-1)          # -1 = value not calculable
sorted_rows = np.where(flagged)[0][np.argsort(-bill_value[flagged])]
register = []
for row in sorted_rows:
    reasons = " | ".join(name for name in problems if problems[name][row])
    register.append(f"{int(ids[row])},{reasons},{bill_value[row]}")
save_file("exception_register.csv", ["transaction_id,reasons,bill_value"] + register)
print("\n".join(register[:10]))

100664,HIGH_PRICE,2096.0
100310,HIGH_PRICE,1296.0
104869,HIGH_PRICE,1168.0
100763,MISSING_COST,460.0
103008,BAD_DISCOUNT,435.0
104616,HIGH_DISCOUNT,415.0
101519,ZERO_STOCK,388.0
104498,BAD_RATING,376.0
103397,HIGH_DISCOUNT,328.0
101117,MISSING_COST,312.0


---
## Phase 9 - Discounts and Scenarios
**46. Discount Band Analysis**

In [110]:
for low, high in [(0, 0), (0.01, 0.09), (0.10, 0.19), (0.20, 0.49), (0.50, 1.0)]:
    band = (discount >= low) & (discount <= high)
    print(low, "-", high, "Bills:", band.sum(), "Avg Qty:", round(quantity[band].mean(), 2),
          "Margin:", round(margin_percent(gross_profit[band].sum(), net_revenue[band].sum()), 2), "%")

0 - 0 Bills: 2508 Avg Qty: 2.5 Margin: 52.38 %
0.01 - 0.09 Bills: 703 Avg Qty: 2.69 Margin: 50.88 %
0.1 - 0.19 Bills: 1282 Avg Qty: 2.97 Margin: 45.86 %
0.2 - 0.49 Bills: 478 Avg Qty: 3.33 Margin: 39.13 %
0.5 - 1.0 Bills: 5 Avg Qty: 3.4 Margin: -81.57 %


**47. Correlation and Covariance**

In [111]:
print("Corr Discount vs Quantity:", round(np.corrcoef(discount, quantity)[0, 1], 3))
print("Corr Discount vs Margin:", round(np.corrcoef(discount, bill_margin)[0, 1], 3))
print("Covariance:\n", np.cov(discount, quantity))

Corr Discount vs Quantity: 0.209
Corr Discount vs Margin: -0.629
Covariance:
 [[0.00588211 0.02159623]
 [0.02159623 1.8120273 ]]


More discount = lower margin. Discount vs quantity link is weak. Correlation does not prove cause.

**48. Flat Discount Scenarios** (not a forecast, quantity and cost same)

In [112]:
rates = np.array([[0], [0.05], [0.10]])                    # 3 scenarios, shape (3, 1)
revenue = (gross_sales * (1 - rates)).sum(axis=1)          # broadcasting
profit = revenue - total_cost.sum()
print("Discount 0%, 5%, 10% -> Revenue:", np.round(revenue), "Profit:", np.round(profit))
print("Profit Change %:", np.round((profit / gross_profit.sum() - 1) * 100, 2))

Discount 0%, 5%, 10% -> Revenue: [1169589. 1111110. 1052630.] Profit: [614553. 556074. 497594.]
Profit Change %: [14.82  3.9  -7.03]


**49. Cap Discount at 10%**

In [113]:
capped_revenue = gross_sales * (1 - np.minimum(discount, 0.10))
capped_profit = capped_revenue.sum() - total_cost.sum()
print("Cap 10% -> Revenue:", round(capped_revenue.sum(), 2), "Profit:", round(capped_profit, 2))
print("Profit Change %:", round((capped_profit / gross_profit.sum() - 1) * 100, 2))

Cap 10% -> Revenue: 1114651.6 Profit: 559615.98
Profit Change %: 4.56


**50. No Discount with Fewer Units** (unit loss is an assumption)

In [114]:
for loss in [0.05, 0.10, 0.20]:
    new_qty = np.where(discount > 0, quantity * (1 - loss), quantity)
    profit = (new_qty * price - new_qty * cost).sum()
    print("Units Lost", int(loss * 100), "% -> Profit:", round(profit, 2))

Units Lost 5 % -> Profit: 597873.69
Units Lost 10 % -> Profit: 581194.0
Units Lost 20 % -> Profit: 547834.62


---
## Phase 10 - Pipeline
**51. Pipeline Function**

In [115]:
def run_pipeline(file_name):
    rows = np.genfromtxt(file_name, delimiter=",", skip_header=1, ndmin=2)
    qty, price, cost, disc = rows[:, 5], rows[:, 6], rows[:, 7], rows[:, 8]
    outlet, product, category = rows[:, 2], rows[:, 3], rows[:, 4]
    valid = (qty > 0) & (price > 0) & (cost >= 0) & (disc >= 0) & (disc <= 1) & np.isin(outlet, [1, 2, 3])
    valid &= np.isin(product, range(1, 9)) & (category == category_of[np.clip(product, 0, 8).astype(int)])
    net = calculate(qty[valid], price[valid], cost[valid], disc[valid])[2]
    return net.sum()

**52. Run Pipeline**

In [116]:
total = run_pipeline("freshflow_sales.csv")
print("Pipeline Revenue:", round(total, 2))
np.testing.assert_allclose(total, net_revenue.sum())

Pipeline Revenue: 1090255.55


**53. Array Operations**

In [117]:
arr = np.arange(1, 13)
print("Reshape:\n", arr.reshape(3, 4), "\nTranspose:\n", arr.reshape(3, 4).T)
print("Ravel:", arr.reshape(3, 4).ravel(), "Sqrt:", np.round(np.sqrt(arr[:4]), 2))
print("Cumsum:", np.cumsum(arr), "Cumprod:", np.cumprod(arr[:5]))

Reshape:
 [[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]] 
Transpose:
 [[ 1  5  9]
 [ 2  6 10]
 [ 3  7 11]
 [ 4  8 12]]
Ravel: [ 1  2  3  4  5  6  7  8  9 10 11 12] Sqrt: [1.   1.41 1.73 2.  ]
Cumsum: [ 1  3  6 10 15 21 28 36 45 55 66 78] Cumprod: [  1   2   6  24 120]


**54. Join and Split Arrays**

In [118]:
print("vstack:\n", np.vstack([outlet_sales, outlet_profit]))
print("hstack:", np.hstack([outlet_sales, outlet_profit]))
print("split (10 days each):", [round(part.sum()) for part in np.split(daily_sales, 3)])
print("array_split sizes:", [len(part) for part in np.array_split(daily_sales, 4)], "| table @ ones:", sales_table @ np.ones(8))

vstack:
 [[438666.4  376609.15 274980.  ]
 [216289.73 183843.71 135086.49]]
hstack: [438666.4  376609.15 274980.   216289.73 183843.71 135086.49]
split (10 days each): [370240, 384941, 335075]
array_split sizes: [8, 8, 7, 7] | table @ ones: [438666.4  376609.15 274980.  ]


**55. Other NumPy Functions**

In [119]:
arr = np.array([10, 20, 30, 40])
print("log1p:", np.round(np.log1p(arr), 2), "max:", np.maximum(arr, 20), "min:", np.minimum(arr, 20), "clip:", np.clip(arr, 15, 35))
print("abs:", np.abs([-5, 3]), "expand_dims:", np.expand_dims(arr, 0).shape, "squeeze:", np.squeeze(np.expand_dims(arr, 0)).shape)
print("isclose:", np.isclose(net_revenue, gross_sales - discount_amount).all())
print("Random IDs:", clean_data[np.random.default_rng(42).choice(len(clean_data), 5, replace=False), 0])

log1p: [2.4  3.04 3.43 3.71] max: [20 20 30 40] min: [10 20 20 20] clip: [15 20 30 35]
abs: [5 3] expand_dims: (1, 4) squeeze: (4,)
isclose: True
Random IDs: [103862. 102190. 103267. 100445. 102161.]


---
## Phase 11 - Tests
**56. Check 5 Rows by Hand**

In [120]:
for i in range(5):
    row_qty, row_price, row_cost, row_disc = clean_data[i, 5:9]
    hand_net = row_qty * row_price - row_qty * row_price * row_disc
    hand_profit = hand_net - row_qty * row_cost
    print("ID", int(clean_data[i, 0]), "Hand:", round(hand_net, 2), "NumPy:", round(net_revenue[i], 2), "| Profit:", round(hand_profit, 2), round(gross_profit[i], 2))
    np.testing.assert_allclose([hand_net, hand_profit], [net_revenue[i], gross_profit[i]])

ID 100001 Hand: 243.2 NumPy: 243.2 | Profit: 123.32 123.32
ID 100002 Hand: 387.0 NumPy: 387.0 | Profit: 176.31 176.31
ID 100003 Hand: 490.0 NumPy: 490.0 | Profit: 285.95 285.95
ID 100004 Hand: 415.8 NumPy: 415.8 | Profit: 191.64 191.64
ID 100005 Hand: 276.0 NumPy: 276.0 | Profit: 151.47 151.47


**57. Test 1 and 2 - Simple Bill and NaN**

In [121]:
gross, disc_amount, net, total, profit = calculate(np.array([2]), np.array([80]), np.array([30]), np.array([0.10]))
np.testing.assert_allclose([net[0], profit[0]], [144, 84])        # 160 - 16 = 144, 144 - 60 = 84
assert np.isnan(calculate(np.array([2]), np.array([np.nan]), np.array([30]), np.array([0]))[2][0])
print("Test 1 passed: simple bill | Test 2 passed: NaN price stays NaN")

Test 1 passed: simple bill | Test 2 passed: NaN price stays NaN


**58. Test 3 and 4 - Zero Denominator and Wrong Shape**

In [122]:
test_net, test_profit = np.array([0.0, 100.0]), np.array([-10.0, 40.0])
assert np.isnan(np.divide(test_profit, test_net, out=np.full(2, np.nan), where=test_net != 0)[0])
try:
    calculate(np.ones(3), np.ones(2), np.ones(3), np.ones(3))
except ValueError:
    print("Test 3 passed: zero denominator | Test 4 passed: wrong shape gives error")

Test 3 passed: zero denominator | Test 4 passed: wrong shape gives error


**59. Test 5 and 6 - Single Row and Raw Data**

In [123]:
one_row = np.genfromtxt(StringIO("h\n1,1,1,1,1,2,80,30,0.1,10,1,5\n"), delimiter=",", skip_header=1, ndmin=2)
assert one_row.shape == (1, 12)
np.testing.assert_array_equal(raw_data, data)
np.testing.assert_allclose(outlet_sales.sum(), net_revenue.sum())
print("Test 5 passed: single row is 2-D | Test 6 passed: raw data safe, totals match")

Test 5 passed: single row is 2-D | Test 6 passed: raw data safe, totals match


**60. Bug Fix:** old code used `category[product_code - 1]`. For product 0 this becomes index -1 and NumPy silently takes the **last** value.
Old code also used `cost > 0`, which wrongly removed cost 0. **Fix:** check the product code first, use `category_of[product]`, and use `cost < 0` for bad cost.

In [124]:
old_category = np.array([1, 1, 1, 1, 2, 2, 3, 4])
print("Bug: product 0 ->", old_category[0 - 1], "(wrong, takes last value)")
print("Fix: product 0 valid? ->", np.isin(0, range(1, 9)), "so it is flagged, not looked up")

Bug: product 0 -> 4 (wrong, takes last value)
Fix: product 0 valid? -> False so it is flagged, not looked up


---
## Phase 12 - Final Report
**61. Final Report**

In [125]:
print("Transactions:", len(data), "Valid:", len(clean_data), "Removed:", invalid.sum())
print("Revenue:", round(net_revenue.sum(), 2), "Profit:", round(gross_profit.sum(), 2), "Margin:", round(margin, 2), "%")
print("Best Outlet:", outlet_names[np.argmax(outlet_sales)], "| Most Units:", product_names[np.argmax(product_units)])
print("Most Profit:", product_names[np.argmax(product_profit)], "| Unusual Days:", unusual_days)

Transactions: 5000 Valid: 4976 Removed: 24
Revenue: 1090255.55 Profit: 535219.93 Margin: 49.09 %
Best Outlet: Outlet A | Most Units: Orange Juice
Most Profit: Mango Juice | Unusual Days: [12 22]


**62. Save Report**

In [126]:
report = [f"Valid Transactions: {len(clean_data)}", f"Revenue: {round(net_revenue.sum(), 2)}", f"Gross Profit: {round(gross_profit.sum(), 2)}",
          f"Margin: {round(margin, 2)} %", f"Best Outlet: {outlet_names[np.argmax(outlet_sales)]}", f"Most Profit: {product_names[np.argmax(product_profit)]}"]
save_file("freshflow_report.txt", report)
print("Report saved to freshflow_report.txt")

Report saved to freshflow_report.txt
